In [0]:
-- Create Data Cube with proper Online/In-Store identification

CREATE OR REPLACE TABLE electronic_retailer.03_Gold.data_cube_sales AS
SELECT 
  d.year,
  d.month,
  d.month_name,
  c.country AS customer_country,
  c.continent AS customer_continent,
  c.gender,
  p.category,
  p.subcategory,
  s.country AS store_country,
  CASE WHEN f.store_key IS NULL THEN 'Online' ELSE 'In-Store' END AS channel,
  
  ROUND(SUM(f.revenue_usd), 2) AS total_revenue,
  SUM(f.quantity) AS total_quantity,
  COUNT(f.order_number) AS order_count,
  ROUND(AVG(f.revenue_usd), 2) AS avg_order_value,
  ROUND(AVG(f.delivery_days), 2) AS avg_delivery_days,
  COUNT(CASE WHEN f.delivery_days IS NOT NULL THEN 1 END) AS delivered_order_count,
  COUNT(CASE WHEN f.delivery_days IS NULL THEN 1 END) AS pending_order_count,
  ROUND(COUNT(CASE WHEN f.delivery_days IS NOT NULL THEN 1 END) * 100.0 / COUNT(*), 2) AS delivery_rate_pct
  
FROM electronic_retailer.03_Gold.fact_sales f
JOIN electronic_retailer.03_Gold.dim_date d ON f.order_date = d.date
JOIN electronic_retailer.03_Gold.dim_customers c ON f.customer_key = c.customer_key
JOIN electronic_retailer.03_Gold.dim_products p ON f.product_key = p.product_key
LEFT JOIN electronic_retailer.03_Gold.dim_stores s ON f.store_key = s.store_key

GROUP BY 
  d.year, d.month, d.month_name,
  c.country, c.continent, c.gender,
  p.category, p.subcategory,
  s.country,
  CASE WHEN f.store_key IS NULL THEN 'Online' ELSE 'In-Store' END

In [0]:
SELECT 
  COUNT(*) AS total_aggregations,
  COUNT(DISTINCT year) AS years,
  COUNT(DISTINCT category) AS categories,
  COUNT(DISTINCT customer_continent) AS continents
FROM electronic_retailer.03_Gold.data_cube_sales
;

select * from electronic_retailer.03_Gold.data_cube_sales limit 50;